## **RAG Chatbot — El Quijote**

A minimal RAG (Retrieval-Augmented Generation) example:
- **Source**: *El ingenioso hidalgo Don Quijote de la Mancha* (PDF, Spanish)
- **Conversation language**: English (forced)
- **Strategy**: translate the user query to Spanish → retrieve relevant chunks → answer in English
- **Stack**: LangChain · ChromaDB · Ollama

In [11]:
# Install dependencies (run once)
# !pip install langchain langchain-community langchain-ollama langchain-chroma chromadb pypdf

In [12]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import ollama
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

CHAT_MODEL      = "qwen3:latest"       # for generation & translation
EMBEDDING_MODEL = "nomic-embed-text"   # dedicated embedding model

PDF_PATH = "./data/El_quijote.pdf"

print(f"Chat model:      {CHAT_MODEL}")
print(f"Embedding model: {EMBEDDING_MODEL}")

Chat model:      qwen3:latest
Embedding model: nomic-embed-text


#### **1. Load and Chunk the PDF**

In [13]:
loader = PyPDFLoader(PDF_PATH)
pages = loader.load()
print(f"Loaded {len(pages)} pages from El Quijote")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " "],
)
chunks = splitter.split_documents(pages)
print(f"Split into {len(chunks)} chunks (avg ~800 chars each)")

Loaded 471 pages from El Quijote
Split into 2100 chunks (avg ~800 chars each)


#### **2. Build the ChromaDB Vector Store**

Embeddings are generated with `nomic-embed-text` via Ollama.  
The store is kept **in-memory** (no disk write needed). This step takes a few minutes.<br>
For a real application you must store them in disk. You can do it using Chroma DB as well

In [14]:
embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL)

print("Building in-memory ChromaDB vector store (this may take a few minutes)...")
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
)
print("Vector store ready.")

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print(f"Collection size: {vectorstore._collection.count()} chunks")

Building in-memory ChromaDB vector store (this may take a few minutes)...
Vector store ready.
Collection size: 2100 chunks


#### **3. RAG Helper Functions**

**Translation strategy**  
The book is in Spanish; the user asks in English. To retrieve meaningful chunks we:
1. Translate the English query → Spanish
2. Retrieve the top-k Spanish chunks
3. Feed the original English question + Spanish context to the LLM with an instruction to answer in English

In [15]:
def translate_to_spanish(text: str) -> str:
    """Translate English text to Spanish using the local LLM."""
    response = ollama.chat(
        model=CHAT_MODEL,
        messages=[
            {
                "role": "system",
                "content": "You are a translator. Translate the user's text to Spanish. Output ONLY the translation, nothing else.",
            },
            {"role": "user", "content": text},
        ],
    )
    return response.message.content.strip()


def ask_quijote(question: str) -> str:
    """
    RAG pipeline:
      1. Translate question to Spanish
      2. Retrieve relevant chunks from ChromaDB
      3. Answer in English using retrieved context
    """
    # Step 1 — translate query for better retrieval
    spanish_query = translate_to_spanish(question)
    print(f"  [translated query] {spanish_query}")

    # Step 2 — retrieve relevant passages
    docs = retriever.invoke(spanish_query)
    context = "\n\n---\n\n".join(d.page_content for d in docs)

    # Step 3 — generate English answer grounded in context
    system_prompt = (
        "You are a literary expert on Don Quixote (El Quijote). "
        "Answer the user's question in English using ONLY the context passages provided. "
        "The context is in Spanish — read it, but always reply in English. "
        "If the context does not contain enough information, say so honestly. "
        "Keep the answer concise (3–5 sentences)."
    )
    user_message = (
        f"Context passages from El Quijote (Spanish):\n{context}\n\n"
        f"Question (in English): {question}"
    )

    response = ollama.chat(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_message},
        ],
    )
    tokens_in  = response.get("prompt_eval_count", "?")
    tokens_out = response.get("eval_count", "?")
    print(f"  [tokens — in: {tokens_in}  out: {tokens_out}]")
    return response.message.content


print("RAG functions ready.")

RAG functions ready.


#### **4. Q&A Demo**

In [16]:
q = "Who is Sancho Panza and what is his relationship with Don Quixote?"
print(f"Q: {q}\n")
print(f"A: {ask_quijote(q)}")

Q: Who is Sancho Panza and what is his relationship with Don Quixote?

  [translated query] Sancho Panza es un personaje central en la novela "Don Quijote de la Mancha" de Miguel de Cervantes. Es el escudero de Don Quijote, un hombre humilde y práctico que se convierte en su compañero de aventuras. Su relación es compleja: aunque inicialmente es un servidor leal, desarrollan una amistad profunda y mutua admiración. Sancho aporta realismo y sensatez a las locuras de Don Quijote, equilibrando su idealismo con la pragmática visión del mundo.
  [tokens — in: 1084  out: 313]
A: Sancho Panza is Don Quixote's loyal squire and companion, serving as both his assistant and confidant. Their relationship is marked by mutual dependence: Sancho provides practical support and humor, while Don Quixote inspires Sancho's idealism and courage. The context shows Sancho admiring Don Quixote's bravery and participating in their adventures, highlighting their dynamic as master and squire.


In [17]:
q = "Why does Don Quixote attack the windmills?"
print(f"Q: {q}\n")
print(f"A: {ask_quijote(q)}")

Q: Why does Don Quixote attack the windmills?

  [translated query] ¿Por qué Don Quijote ataca los molinos de viento?
  [tokens — in: 1036  out: 271]
A: Don Quixote attacks the windmills because he mistakenly believes they are giants, a delusion fueled by his obsession with chivalric ideals. He sees them as enemies to be vanquished, driven by his desire for glory and his conviction that the sorcerer Frestón transformed them into windmills to thwart his heroic feats. His attack stems from a mix of madness, imagination, and a misguided quest to combat perceived evil.


In [19]:
q = "Who is Dulcinea del Toboso and what role does she play in the story?"
print(f"Q: {q}\n")
print(f"A: {ask_quijote(q)}")

Q: Who is Dulcinea del Toboso and what role does she play in the story?

  [translated query] ¿Quién es Dulcinea del Toboso y cuál es su papel en la historia?
  [tokens — in: 509  out: 383]
A: Dulcinea del Toboso is Don Quixote's idealized love interest and symbol of his chivalric delusions. She is portrayed as a noblewoman whose name and title are central to his romanticized worldview. Sancho Panza's correction about her being called "Dulcinea del Toboso" rather than "Don Dulcinea" highlights her significance in the narrative, though the text does not elaborate on her character beyond her role as a catalyst for Quixote's fantasies. Her presence underscores the theme of illusion versus reality in the story.


#### **5. Interactive Loop**

Type any question about El Quijote. Type `quit` to exit.

In [18]:
print("=" * 55)
print("  El Quijote RAG — ask anything about the book!")
print("  (type 'quit' to exit)")
print("=" * 55)

while True:
    question = input("\nYou: ").strip()
    if not question:
        continue
    if question.lower() in {"quit", "exit"}:
        print("Goodbye!")
        break
    print()
    answer = ask_quijote(question)
    print(f"RAG: {answer}")

  El Quijote RAG — ask anything about the book!
  (type 'quit' to exit)



You:  in which chapter Don Qixote attacks the windmills



  [translated query] en qué capítulo Don Quijote ataca a las molinos de viento
  [tokens — in: 1036  out: 242]
RAG: Don Quixote attacks the windmills in **Chapter VIII** ("Del buen suceso que el valeroso don Quijote tuvo en la espantable y jamás imaginada aventura de los molinos de viento"). This chapter details his mistaken battle with the windmills, which he perceives as giants, and Sancho Panza's attempts to correct him.



You:  who is the father of don qixote



  [translated query] ¿Quién es el padre de Don Quijote?
  [tokens — in: 951  out: 357]
RAG: The provided context passages do not explicitly mention the name of Don Quixote's father. The text refers to "mi señor" and "su padre" in the context of other characters (e.g., don Luis and doña Clara), but no direct information about Don Quixote's father is given. Thus, the context does not answer this question.


...


  [translated query] ¿Qué información puedes encontrar en el libro sobre el autor del libro?
  [tokens — in: 883  out: 410]
RAG: The text refers to the book's author as a "loco y de perro" (madman and dog), mocking their sanity or work. It also accuses the author of usurping Don Quixote's name and annulling his deeds, framing them as a fraud. Additionally, the author is noted to use Aragonés dialect, and their work is described as containing personal grievances and emotional expressions, suggesting a blend of self-referential critique and literary style.



You:  does he say the name of the author?



  [translated query] ¿Dice él el nombre del autor?
  [tokens — in: 908  out: 390]
RAG: In the context, Sancho Panza mentions the author's name as "Cide Hamete Berenjena," and Don Quijote comments on the name being Moorish. However, Don Quijote himself does not explicitly state the author's name; Sancho is the one who reveals it. Thus, the name is mentioned by Sancho, not directly by Don Quijote.



You:  exit


Goodbye!
